In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
import kagglehub
import os
import shutil

# Download latest version
path = kagglehub.dataset_download("andradaolteanu/gtzan-dataset-music-genre-classification")

print("Dataset downloaded to:", path)
print("PyTorch version:", torch.__version__)
print("Torchvision version:", torchvision.__version__)

In [ ]:
import os
import shutil
import random

# 1. Define paths
source_dir = '/kaggle/input/gtzan-dataset-music-genre-classification/Data/genres_original'
target_root = 'gtzan_split'
splits = ['train', 'val', 'test']

# 2. Create directory structure
genres = [d for d in os.listdir(source_dir) if os.path.isdir(os.path.join(source_dir, d))]
for split in splits:
    for genre in genres:
        os.makedirs(os.path.join(target_root, split, genre), exist_ok=True)

# 3-5. Iterate, Split, and Copy
random.seed(42)  # For reproducibility
stats = {split: 0 for split in splits}

for genre in genres:
    genre_path = os.path.join(source_dir, genre)
    files = [f for f in os.listdir(genre_path) if f.endswith('.wav')]
    random.shuffle(files)

    n = len(files)
    train_idx = int(n * 0.8)
    val_idx = int(n * 0.9)

    file_splits = {
        'train': files[:train_idx],
        'val': files[train_idx:val_idx],
        'test': files[val_idx:]
    }

    for split, split_files in file_splits.items():
        for f in split_files:
            src = os.path.join(genre_path, f)
            dst = os.path.join(target_root, split, genre, f)
            shutil.copy(src, dst)
            stats[split] += 1

# 6. Print Verification
print("Data split complete.")
for split, count in stats.items():
    print(f"{split.capitalize()} set: {count} files")

In [ ]:
import torch
import torchaudio
import os
from torch.utils.data import Dataset
import torch.nn.functional as F

class GTZANDataset(Dataset):
    def __init__(self, root_dir, split, sample_rate=22050, duration=15):
        self.root_dir = os.path.join(root_dir, split)
        self.sample_rate = sample_rate
        self.duration = duration
        self.n_samples = sample_rate * duration
        self.file_list = []

        # Map genres to integers
        genres = sorted([d for d in os.listdir(self.root_dir) if os.path.isdir(os.path.join(self.root_dir, d))])
        self.label_to_idx = {genre: i for i, genre in enumerate(genres)}

        for genre in genres:
            genre_dir = os.path.join(self.root_dir, genre)
            for f in os.listdir(genre_dir):
                if f.endswith('.wav'):
                    full_path = os.path.join(genre_dir, f)
                    # We split each ~30s file into two segments of 'duration' (15s)
                    # Segment 0: 0s to 15s, Segment 1: 15s to 30s
                    self.file_list.append((full_path, self.label_to_idx[genre], 0))
                    self.file_list.append((full_path, self.label_to_idx[genre], 1))

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        path, label, segment_idx = self.file_list[idx]

        # Calculate offset in samples
        offset = segment_idx * self.n_samples

        # Load specific segment using frame parameters to save memory/time
        waveform, sr = torchaudio.load(path, frame_offset=offset, num_frames=self.n_samples)

        # Resample if necessary
        if sr != self.sample_rate:
            resampler = torchaudio.transforms.Resample(orig_freq=sr, new_freq=self.sample_rate)
            waveform = resampler(waveform)

        # Convert to mono if stereo
        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)

        # Ensure consistent sequence length (padding/truncating if source is slightly off)
        if waveform.shape[1] < self.n_samples:
            padding = self.n_samples - waveform.shape[1]
            waveform = F.pad(waveform, (0, padding))
        else:
            waveform = waveform[:, :self.n_samples]

        # Normalization
        if waveform.abs().max() > 0:
            waveform = waveform / waveform.abs().max()

        return waveform.squeeze(0), torch.tensor(label)


In [ ]:
from torch.utils.data import DataLoader

# 1. Initialize Datasets using the updated torchaudio-based class
train_dataset = GTZANDataset(root_dir='gtzan_split', split='train')
val_dataset = GTZANDataset(root_dir='gtzan_split', split='val')
test_dataset = GTZANDataset(root_dir='gtzan_split', split='test')

# 2. Initialize DataLoaders
batch_size = 16
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# 3. Verify the pipeline
audios, labels = next(iter(train_loader))

print(f'Train Loader - Batch Audio Shape: {audios.shape}')
print(f'Train Loader - Batch Labels: {labels}')
print(f'Dataset Size - Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}')
print('DataLoaders are updated and ready.')